# 20 — Cloud Management: Governance, Cost, Monitoring & DevSecOps

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Cloud Security & Operations  
**Certifications:** AWS Certified Security – Specialty, CompTIA CySA+

---

## Objectives
- Build a cloud governance framework with tagging, policies, and guardrails
- Implement FinOps — cloud cost management and optimisation
- Design a cloud monitoring and observability strategy
- Apply Infrastructure as Code (IaC) security scanning
- Build a DevSecOps pipeline with security gates
- Manage multi-cloud environments
- Generate a cloud management maturity report

## 1. Setup & Imports

In [ ]:
import json
import re
from datetime import datetime, timezone
from collections import Counter, defaultdict

## 2. Cloud Governance Framework

In [ ]:
# Cloud governance = policies + guardrails + visibility
# Simulate a resource tagging compliance check
REQUIRED_TAGS = ['Environment', 'Owner', 'CostCentre', 'Project', 'DataClassification']

CLOUD_RESOURCES = [
    {'id': 'i-0abc123', 'type': 'EC2',   'tags': {'Environment': 'prod', 'Owner': 'bency', 'CostCentre': 'CC-101', 'Project': 'webapp', 'DataClassification': 'Confidential'}},
    {'id': 'i-0def456', 'type': 'EC2',   'tags': {'Environment': 'dev', 'Owner': 'jsmith'}},
    {'id': 'bucket-1',  'type': 'S3',    'tags': {'Environment': 'prod', 'Owner': 'team-data', 'CostCentre': 'CC-102', 'Project': 'analytics', 'DataClassification': 'Internal'}},
    {'id': 'db-main',   'type': 'RDS',   'tags': {'Environment': 'prod'}},
    {'id': 'fn-worker', 'type': 'Lambda','tags': {'Environment': 'prod', 'Owner': 'bency', 'CostCentre': 'CC-101', 'Project': 'webapp', 'DataClassification': 'Internal'}},
    {'id': 'sg-abc',    'type': 'SG',    'tags': {}},
]

def check_tag_compliance(resource: dict, required: list) -> dict:
    tags    = resource.get('tags', {})
    missing = [t for t in required if t not in tags]
    score   = (len(required) - len(missing)) / len(required) * 100
    return {'missing_tags': missing, 'compliance_score': round(score, 0), 'compliant': len(missing) == 0}


print('=== CLOUD RESOURCE TAGGING COMPLIANCE ===\n')
print(f'{"Resource ID":<14} {"Type":<8} {"Score":<8} Status     Missing Tags')
print('-' * 75)
violations = []
for r in CLOUD_RESOURCES:
    result = check_tag_compliance(r, REQUIRED_TAGS)
    status = '✅ COMPLIANT' if result['compliant'] else '❌ VIOLATION'
    if not result['compliant']:
        violations.append({'resource': r['id'], 'missing': result['missing_tags']})
    print(f"  {r['id']:<14} {r['type']:<8} {result['compliance_score']:<7.0f}%  {status}   {result['missing_tags']}")

print(f'\nTag compliance rate: {len(CLOUD_RESOURCES)-len(violations)}/{len(CLOUD_RESOURCES)} resources fully tagged')

# Service Control Policy (SCP) examples
print('\n=== GOVERNANCE GUARDRAILS (SCP Examples) ===')
SCPS = [
    'Deny all API calls outside approved regions (us-east-1, eu-west-1)',
    'Deny creation of unencrypted S3 buckets',
    'Deny disabling of CloudTrail or GuardDuty',
    'Require MFA for all IAM user console access',
    'Deny root account usage for any API call',
]
for scp in SCPS:
    print(f'  🔒 {scp}')

## 3. FinOps — Cloud Cost Management

In [ ]:
# Simulated monthly cloud spend by service and environment
MONTHLY_SPEND = [
    {'service': 'EC2',        'env': 'prod',    'cost': 12400, 'optimised': False},
    {'service': 'EC2',        'env': 'dev',     'cost': 3200,  'optimised': False},
    {'service': 'RDS',        'env': 'prod',    'cost': 5800,  'optimised': True},
    {'service': 'S3',         'env': 'prod',    'cost': 890,   'optimised': True},
    {'service': 'CloudFront', 'env': 'prod',    'cost': 420,   'optimised': True},
    {'service': 'Lambda',     'env': 'prod',    'cost': 180,   'optimised': True},
    {'service': 'EKS',        'env': 'prod',    'cost': 4200,  'optimised': False},
    {'service': 'EKS',        'env': 'staging', 'cost': 2100,  'optimised': False},
    {'service': 'Data Transfer','env': 'prod',  'cost': 1600,  'optimised': False},
    {'service': 'Idle EC2',   'env': 'dev',     'cost': 780,   'optimised': False},  # Waste!
]

total_spend  = sum(r['cost'] for r in MONTHLY_SPEND)
by_service   = Counter({r['service']: 0 for r in MONTHLY_SPEND})
for r in MONTHLY_SPEND:
    by_service[r['service']] += r['cost']

OPTIMISATIONS = [
    {'action': 'Purchase Savings Plans for prod EC2 (1-year, no upfront)', 'saving': 3720, 'effort': 'LOW'},
    {'action': 'Right-size over-provisioned EC2 instances (m5.2xl → m5.xl)', 'saving': 2480, 'effort': 'MEDIUM'},
    {'action': 'Use Spot Instances for EKS worker nodes (dev/staging)',    'saving': 1470, 'effort': 'MEDIUM'},
    {'action': 'Delete idle EC2 instances in dev environment',             'saving': 780,  'effort': 'LOW'},
    {'action': 'Move S3 old objects to Intelligent-Tiering',               'saving': 267,  'effort': 'LOW'},
    {'action': 'Use VPC Endpoints to reduce data transfer costs',          'saving': 480,  'effort': 'LOW'},
]

total_saving = sum(o['saving'] for o in OPTIMISATIONS)

print('=== MONTHLY CLOUD SPEND ANALYSIS ===\n')
print(f'Total Monthly Spend: ${total_spend:,.0f}\n')
print('By Service:')
for svc, cost in sorted(by_service.items(), key=lambda x: -x[1]):
    pct = cost / total_spend * 100
    bar = '█' * int(pct / 3)
    print(f'  {svc:<15} ${cost:>6,}  ({pct:4.1f}%)  {bar}')

print(f'\n=== COST OPTIMISATION OPPORTUNITIES ===\n')
print(f'{"Action":<60} {"Saving/mo":<12} Effort')
print('-' * 80)
for o in sorted(OPTIMISATIONS, key=lambda x: -x['saving']):
    print(f"  {o['action']:<58} ${o['saving']:>5,}/mo  {o['effort']}")
print(f'\nTotal potential savings: ${total_saving:,}/mo  (${total_saving*12:,}/yr)')
print(f'Savings as % of spend : {total_saving/total_spend*100:.0f}%')

## 4. Cloud Monitoring & Observability

In [ ]:
# The Three Pillars of Observability: Metrics, Logs, Traces
OBSERVABILITY_PILLARS = {
    'Metrics': {
        'description': 'Numerical measurements over time',
        'examples'   : ['CPU utilisation', 'Request latency p99', 'Error rate', 'Disk IOPS'],
        'aws_service': 'CloudWatch Metrics',
        'alert_on'   : 'CPU > 80% for 5min | Error rate > 1% | Latency > 2s'
    },
    'Logs': {
        'description': 'Timestamped records of events',
        'examples'   : ['Application errors', 'Access logs', 'CloudTrail API events', 'VPC Flow Logs'],
        'aws_service': 'CloudWatch Logs / S3 + Athena',
        'alert_on'   : 'ERROR count spike | Failed login > 5 | Unauthorized API call'
    },
    'Traces': {
        'description': 'End-to-end request paths across distributed services',
        'examples'   : ['API Gateway → Lambda → DynamoDB trace', 'Microservice call chains'],
        'aws_service': 'AWS X-Ray',
        'alert_on'   : 'Trace error rate > 2% | Service latency anomaly'
    }
}

# Simulated CloudWatch alarms
CW_ALARMS = [
    {'name': 'High-CPU-Prod',          'metric': 'CPUUtilization',      'threshold': '>80%',  'status': 'OK',     'action': 'SNS → PagerDuty'},
    {'name': 'RDS-FreeStorage-Low',    'metric': 'FreeStorageSpace',    'threshold': '<10GB', 'status': 'ALARM',  'action': 'SNS → On-call'},
    {'name': 'Lambda-Error-Rate',      'metric': 'Errors',              'threshold': '>1%',   'status': 'OK',     'action': 'SNS → Slack'},
    {'name': 'UnauthorizedAPICalls',   'metric': 'CloudTrail filter',   'threshold': '>0',    'status': 'ALARM',  'action': 'SNS → Security team'},
    {'name': 'Billing-Threshold',      'metric': 'EstimatedCharges',    'threshold': '>$35K', 'status': 'OK',     'action': 'SNS → Finance'},
    {'name': 'GuardDuty-High-Finding', 'metric': 'GuardDuty findings',  'threshold': 'HIGH',  'status': 'ALARM',  'action': 'SNS → SIEM + IR team'},
]

print('=== CLOUD OBSERVABILITY STRATEGY ===\n')
for pillar, details in OBSERVABILITY_PILLARS.items():
    print(f'  [{pillar}] — {details["description"]}')
    print(f'    Service  : {details["aws_service"]}')
    print(f'    Alert on : {details["alert_on"]}\n')

print('Active CloudWatch Alarms:')
for alarm in CW_ALARMS:
    icon = '🔴' if alarm['status'] == 'ALARM' else '✅'
    print(f"  {icon} {alarm['name']:<30} [{alarm['status']:<5}]  Threshold: {alarm['threshold']:<8}  → {alarm['action']}")

## 5. Infrastructure as Code Security (IaC Scanning)

In [ ]:
# Simulate scanning CloudFormation / Terraform for misconfigurations
# (mirrors what tools like Checkov, cfn-nag, tfsec do)
SAMPLE_IAC_CONFIGS = [
    {
        'resource': 'aws_s3_bucket.data',
        'config'  : {'versioning': False, 'encryption': False, 'public_access_block': False, 'logging': False}
    },
    {
        'resource': 'aws_security_group.web',
        'config'  : {'ingress_0_0_0_0_22': True, 'ingress_0_0_0_0_3389': True, 'egress_all': True}
    },
    {
        'resource': 'aws_db_instance.main',
        'config'  : {'storage_encrypted': False, 'publicly_accessible': True, 'backup_retention': 0, 'deletion_protection': False}
    },
    {
        'resource': 'aws_lambda_function.processor',
        'config'  : {'tracing': True, 'vpc_config': True, 'reserved_concurrency': 100}
    },
]

IAC_RULES = [
    ('S3 encryption must be enabled',            lambda c: c.get('encryption', True),              'HIGH'),
    ('S3 versioning must be enabled',            lambda c: c.get('versioning', True),               'MEDIUM'),
    ('S3 public access block required',          lambda c: c.get('public_access_block', True),      'CRITICAL'),
    ('SSH (22) must not be open to 0.0.0.0/0',  lambda c: not c.get('ingress_0_0_0_0_22', False),  'CRITICAL'),
    ('RDP (3389) must not be open to 0.0.0.0/0',lambda c: not c.get('ingress_0_0_0_0_3389', False),'CRITICAL'),
    ('RDS must be encrypted',                    lambda c: c.get('storage_encrypted', True),        'CRITICAL'),
    ('RDS must not be publicly accessible',      lambda c: not c.get('publicly_accessible', False), 'CRITICAL'),
    ('RDS backup retention must be >= 7 days',   lambda c: c.get('backup_retention', 7) >= 7,       'HIGH'),
]

iac_findings = []
print('=== IAC SECURITY SCAN RESULTS ===\n')
for resource in SAMPLE_IAC_CONFIGS:
    resource_findings = []
    for rule_desc, check_fn, severity in IAC_RULES:
        try:
            if not check_fn(resource['config']):
                resource_findings.append({'severity': severity, 'rule': rule_desc})
        except Exception:
            pass
    status = f'{len(resource_findings)} issue(s)' if resource_findings else '✅ Clean'
    print(f"  Resource: {resource['resource']}  →  {status}")
    for f in resource_findings:
        iac_findings.append({**f, 'resource': resource['resource']})
        print(f"    [{f['severity']:8}] {f['rule']}")

print(f'\nTotal IaC findings: {len(iac_findings)}')
print('Tools that automate this: Checkov, tfsec, cfn-nag, Terrascan, KICS')

## 6. DevSecOps Pipeline

In [ ]:
# Simulated CI/CD pipeline with security gates
PIPELINE_STAGES = [
    {
        'stage'   : '1. Source Control',
        'tools'   : ['GitHub', 'Branch protection rules', 'Signed commits'],
        'security': ['Secrets scanning (git-secrets / Gitleaks)', 'Dependency review on PR'],
        'gate'    : 'Block commit if secrets detected'
    },
    {
        'stage'   : '2. Build',
        'tools'   : ['GitHub Actions', 'Docker build'],
        'security': ['SAST — static analysis (Bandit for Python, Semgrep)', 'SCA — dependency CVE scan (Snyk, Dependabot)'],
        'gate'    : 'Fail build on CRITICAL/HIGH CVEs or SAST findings'
    },
    {
        'stage'   : '3. Test',
        'tools'   : ['pytest', 'DAST (OWASP ZAP)'],
        'security': ['Unit & integration tests', 'DAST against staging environment', 'Container image scan (Trivy)'],
        'gate'    : 'Block deployment if DAST finds injection or auth issues'
    },
    {
        'stage'   : '4. IaC Validation',
        'tools'   : ['Terraform plan', 'Checkov', 'OPA / Sentinel'],
        'security': ['IaC misconfiguration scan (Checkov)', 'Policy-as-code validation (OPA)', 'Cost estimation'],
        'gate'    : 'Block if IaC violates security policies'
    },
    {
        'stage'   : '5. Deploy',
        'tools'   : ['Terraform apply', 'ArgoCD', 'AWS CodeDeploy'],
        'security': ['Least-privilege deployment role', 'Blue/green or canary deployment', 'Automated rollback on error'],
        'gate'    : 'Manual approval gate for production deployments'
    },
    {
        'stage'   : '6. Monitor & Respond',
        'tools'   : ['CloudWatch', 'GuardDuty', 'Falco', 'SIEM'],
        'security': ['Runtime threat detection', 'Continuous compliance (AWS Config)', 'SBOM generation'],
        'gate'    : 'Auto-rollback on security alarm; alert IR team'
    },
]

print('=== DEVSECOPS PIPELINE ===\n')
for stage in PIPELINE_STAGES:
    print(f"  ▶ {stage['stage']}")
    print(f"    Tools   : {', '.join(stage['tools'])}")
    print(f"    Security: {' | '.join(stage['security'][:2])}")
    print(f"    🚦 Gate  : {stage['gate']}\n")

## 7. Cloud Management Maturity Report

In [ ]:
MATURITY_DIMENSIONS = [
    {'dimension': 'Governance & Tagging',    'score': 2, 'target': 4, 'notes': 'Tagging policy exists but 4/6 resources non-compliant'},
    {'dimension': 'Cost Management (FinOps)','score': 2, 'target': 4, 'notes': f'${total_saving:,}/mo savings opportunity identified'},
    {'dimension': 'Security Posture',        'score': 3, 'target': 4, 'notes': 'CSPM 6/10 controls passing; root MFA still missing'},
    {'dimension': 'Monitoring & Alerting',   'score': 3, 'target': 4, 'notes': '2 active alarms; missing application tracing'},
    {'dimension': 'IaC & Automation',        'score': 2, 'target': 4, 'notes': 'IaC in use but security scanning not in pipeline yet'},
    {'dimension': 'DevSecOps',               'score': 2, 'target': 4, 'notes': 'SAST and container scanning planned; not yet deployed'},
    {'dimension': 'Incident Response',       'score': 2, 'target': 4, 'notes': 'Cloud IR runbooks not yet created'},
]

total = sum(d['score'] for d in MATURITY_DIMENSIONS)
max_t = len(MATURITY_DIMENSIONS) * 5
pct   = total / max_t * 100

cloud_mgmt_report = {
    'report_generated'   : datetime.now(timezone.utc).isoformat(),
    'assessed_by'        : 'Bency (Benjamin Adjei)',
    'maturity_score'     : f'{total}/{max_t} ({pct:.0f}%)',
    'maturity_level'     : 'DEVELOPING' if pct < 60 else 'DEFINED' if pct < 75 else 'MANAGED',
    'monthly_spend'      : f'${total_spend:,}',
    'savings_opportunity': f'${total_saving:,}/mo',
    'iac_findings'       : len(iac_findings),
    'tagging_violations' : len(violations),
    'top_priorities'     : [
        'Implement mandatory tag enforcement via AWS Config Rules (Governance)',
        f'Purchase Savings Plans — save ${OPTIMISATIONS[0]["saving"]:,}/mo (FinOps)',
        'Integrate Checkov into CI/CD pipeline to catch IaC issues pre-deployment',
        'Build cloud incident response runbooks for GuardDuty finding types',
        'Enable VPC Flow Logs and X-Ray tracing for full observability',
        'Adopt FinOps practice — weekly cost review cadence with engineering leads'
    ],
    'maturity_by_dimension': [
        {'dimension': d['dimension'], 'score': d['score'], 'target': d['target'],
         'gap': d['target'] - d['score'], 'notes': d['notes']}
        for d in MATURITY_DIMENSIONS
    ]
}

print('=== CLOUD MANAGEMENT MATURITY ===\n')
print(f'{"Dimension":<28} Score  Target  Gap   Notes')
print('-' * 90)
for d in MATURITY_DIMENSIONS:
    gap  = d['target'] - d['score']
    bar  = '█' * d['score'] + '░' * (5 - d['score'])
    print(f"  {d['dimension']:<26} {d['score']}/5   {d['target']}/5   -{gap}    {d['notes'][:45]}")

print(f'\nMaturity: {total}/{max_t} ({pct:.0f}%) — {cloud_mgmt_report["maturity_level"]}')
print('\n' + json.dumps(cloud_mgmt_report, indent=2))

## 8. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Governance first** | Without tagging and SCPs, cost and security accountability is impossible |
| **FinOps is continuous** | Cloud spend requires weekly review — set budgets and anomaly alerts from day one |
| **Observability trinity** | Metrics + Logs + Traces together — any two without the third leaves blind spots |
| **Security shifts left** | IaC scanning in CI/CD catches misconfigurations before they reach production |
| **DevSecOps gates** | Each pipeline stage should have a security gate — not just a final security review |
| **Automate compliance** | AWS Config + Security Hub + Checkov = continuous compliance — not point-in-time audits |

### Cloud Management Tool Reference
| Tool | Purpose |
|------|---------|
| **AWS Cost Explorer + Budgets** | FinOps — spend visibility and alerts |
| **AWS Config** | Continuous compliance monitoring |
| **Checkov / tfsec** | IaC static analysis |
| **Terraform + Atlantis** | IaC with GitOps workflow |
| **Datadog / Grafana** | Multi-cloud observability |
| **Cloud Custodian** | Policy-as-code for automated resource governance |
| **Komiser / Infracost** | Cloud cost breakdown and forecasting |

---
*🎓 Cloud Series Complete — Notebooks 18, 19, 20 cover Cloud Introduction, Security, and Management*